# 03 · Quality control & perturbation efficacy
1. Select a development subset (random perturbations; controls capped).
2. Cell/gene QC; normalise + log1p (only if X holds raw counts).
3. Measure **on-target knockdown** and drop perturbations without evidence of target suppression.
4. Select genes (top-variance ∪ all kept perturbation targets) so every perturbation can be encoded.
5. Save to Drive: `interim/*_qc.h5ad`, `processed/*_subset.h5ad`.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
from perturbai.data.loaders import load_dataset, looks_like_counts
from perturbai.preprocessing.qc import (choose_perturbations, subsample_cells, cell_qc_metrics, filter_cells_genes,
                                        normalize_log1p_matrix, select_genes)
from perturbai.features.effects import knockdown_efficiency
S, Q = CFG["subset"], CFG["qc"]
adata = load_dataset(DS, P, synthetic=SYNTHETIC)
labels = adata.obs[PERT].astype(str).values
n_pool = int(S["n_perturbations"] * 1.5)       # over-sample, then filter by knockdown
cand = choose_perturbations(labels, CTRL, n_pool, S["min_cells_per_perturbation"] if not SYNTHETIC else 20,
                            seed=CFG["seed"], candidates=set(adata.var_names))
mask = subsample_cells(labels, CTRL, cand, S["max_control_cells"], S["max_cells_per_perturbation"], seed=CFG["seed"])
adata = adata[mask].copy()
print("subset:", adata.shape, "| candidate perturbations:", len(cand))

In [ ]:
counts_like = looks_like_counts(adata.X)
print("raw counts detected:", counts_like)
if counts_like:
    adata.layers["counts"] = adata.X.copy()
    cell_qc_metrics(adata)
    adata = filter_cells_genes(adata, Q["min_genes_per_cell"] if not SYNTHETIC else 20,
                               Q["min_cells_per_gene"], Q["max_pct_mito"])
    Xn, Xl = normalize_log1p_matrix(adata.layers["counts"] if "counts" in adata.layers else adata.X)
else:   # assume log1p(normalised) already
    Xl = adata.X; Xn = Xl.copy(); Xn.data = np.expm1(Xn.data) if sp.issparse(Xn) else np.expm1(Xn)
adata.layers["norm"] = Xn; adata.X = Xl
print("after QC:", adata.shape)

## On-target knockdown

In [ ]:
labels = adata.obs[PERT].astype(str).values
kd = knockdown_efficiency(adata.layers["norm"], labels, adata.var_names, CTRL, Q["min_target_expression"])
kd.to_csv(P.interim / f"{TAG}_knockdown.csv")
print(kd.assessable.value_counts())
plt.hist(kd.knockdown[kd.assessable].dropna(), bins=40); plt.axvline(Q["min_knockdown"], c="r", ls="--")
plt.xlabel("knockdown fraction"); plt.ylabel("# perturbations"); plt.show()
keep = kd.index[(kd.assessable & (kd.knockdown >= Q["min_knockdown"])) | ((~kd.assessable) & Q["keep_unassessable"])]
keep = list(keep)[: S["n_perturbations"]]
print(f"kept {len(keep)} / {len(kd)} perturbations (knockdown >= {Q['min_knockdown']})")

In [ ]:
adata = adata[adata.obs[PERT].astype(str).isin(keep + [CTRL])].copy()
adata.write_h5ad(P.interim / artifact("qc_adata", TAG), compression="gzip")
mask_g = select_genes(adata.X, adata.var_names, S["n_hvg"], force_include=keep)
adata = adata[:, mask_g].copy()
adata.obs[PERT] = adata.obs[PERT].astype(str).astype("category")
adata.write_h5ad(P.processed / artifact("subset_adata", TAG), compression="gzip")
print("final subset:", adata.shape, "| perturbations:", adata.obs[PERT].nunique() - 1)

**Interpretation:** how many perturbations were dropped for weak knockdown? Do dropped perturbations have fewer cells? Record this in your report.